# 02. 타깃 정의 점검
손계산 예시와 함수 출력을 대조합니다 (자동 테스트: `pytest tests/test_targets.py`).

In [ ]:
import sys, pathlib, warnings
cwd = pathlib.Path.cwd()          # 커널 cwd 가 notebooks/ 든 프로젝트 루트든 상관없이 루트를 찾는다
ROOT = next(p for p in [cwd, *cwd.parents] if (p / "configs").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 100

In [ ]:
from src.targets.mdd import trailing_mdd
p = [100, 110, 99, 105, 90]
print("MDD(5) 함수 =", trailing_mdd(p, 5)[-1], "| 손계산 90/110-1 =", 90 / 110 - 1)
print("MDD(3) 함수 =", np.round(trailing_mdd(p, 3), 4), "| 손계산 [nan nan -0.1 -0.1 -0.1429]")

In [ ]:
from src.data.process import load_processed
d = load_processed("KOSPI")
d[["close", "m5", "m20", "yz20"]].tail(8)

In [ ]:
# 라벨 정의: 'origin t 이후 h일의 지표' == series[t+h]
s = d["m20"]; t = 3000; h = 20
print(d.index[t].date(), "->", d.index[t + h].date(), "| y_true =", round(s.iloc[t + h], 4))

# 해당 구간의 가격과 trailing MDD
w = d.iloc[t - 40: t + h + 5]
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
w["close"].plot(ax=ax[0]); ax[0].axvline(d.index[t], color="gray", ls="--"); ax[0].axvline(d.index[t + h], color="red", ls="--")
w["m20"].plot(ax=ax[1]); ax[1].set_ylabel("m20 (trailing)"); ax[0].set_title("origin(gray) -> target(red)")
fig.tight_layout()

In [ ]:
# 2008 / 2020 위기 구간에서 MDD 와 변동성이 같이 움직이는지
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for a, (s, e) in zip(ax, [("2008-06-01", "2009-06-30"), ("2020-01-15", "2020-08-31")]):
    x = d.loc[s:e]
    x["m20"].plot(ax=a, color="tab:red", label="m20"); a.set_ylabel("m20")
    x["yz20"].plot(ax=a.twinx(), color="tab:blue", label="yz20"); a.set_title(f"{s} ~ {e}")
fig.tight_layout()